<a href="https://colab.research.google.com/github/GAIA-HKUSTGZ/UCUG1002_AI-Society/blob/main/labs/2_Network/lab_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI and Society: Network Lab

This Colab connects five ideas in one workflow:

1. Weighted graphs and Dijkstra shortest paths
2. Social network centrality, communities, and homophily
3. Threshold diffusion
4. A tiny two-layer GCN written from scratch in PyTorch
5. A toy graph agent that explains an answer with an evidence path


In [ ]:
# Install/import dependencies. Colab usually has most of these already.
import importlib, subprocess, sys

required = {
    "networkx": "networkx",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "torch": "torch",
}

for module, package in required.items():
    try:
        importlib.import_module(module)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import heapq
import math
import random
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## Part 1: Build a weighted campus graph

Nodes are places. Edges are paths. Edge weights are travel times.


In [ ]:
G_city = nx.Graph()
edges = [
    ("Dorm", "Library", 6),
    ("Dorm", "Cafe", 4),
    ("Library", "Lab", 5),
    ("Library", "Cafe", 3),
    ("Cafe", "Clinic", 7),
    ("Lab", "Clinic", 4),
    ("Lab", "Gate", 8),
    ("Clinic", "Gate", 5),
]
G_city.add_weighted_edges_from(edges, weight="minutes")

pos_city = {
    "Dorm": (0, 1),
    "Library": (1.4, 2.2),
    "Cafe": (1.5, 0.2),
    "Lab": (3.0, 1.7),
    "Clinic": (4.2, 0.8),
    "Gate": (5.3, 2.2),
}

def draw_city_graph(path_edges=None, title="Campus network"):
    path_edges = set(tuple(sorted(e)) for e in (path_edges or []))
    edge_colors = ["tomato" if tuple(sorted((u, v))) in path_edges else "#555555" for u, v in G_city.edges()]
    widths = [4 if tuple(sorted((u, v))) in path_edges else 1.6 for u, v in G_city.edges()]
    nx.draw_networkx_nodes(G_city, pos_city, node_size=1050, node_color="#F6D365", edgecolors="#222222")
    nx.draw_networkx_labels(G_city, pos_city, font_size=10)
    nx.draw_networkx_edges(G_city, pos_city, edge_color=edge_colors, width=widths)
    labels = nx.get_edge_attributes(G_city, "minutes")
    nx.draw_networkx_edge_labels(G_city, pos_city, edge_labels=labels, font_size=9)
    plt.title(title)
    plt.axis("off")
    plt.show()

draw_city_graph()


## Part 2: Implement Dijkstra step by step

Watch the dataframe: each row shows one settled node plus any distance updates.


In [ ]:
def dijkstra_with_steps(G, source, weight="minutes"):
    dist = {node: math.inf for node in G.nodes()}
    parent = {node: None for node in G.nodes()}
    dist[source] = 0
    pq = [(0, source)]
    settled = set()
    steps = []

    while pq:
        current_dist, u = heapq.heappop(pq)
        if u in settled:
            continue
        settled.add(u)

        updates = []
        for v, attrs in G[u].items():
            new_dist = current_dist + attrs[weight]
            if new_dist < dist[v]:
                dist[v] = new_dist
                parent[v] = u
                heapq.heappush(pq, (new_dist, v))
                updates.append(f"{v}={new_dist}")

        steps.append({
            "current": u,
            "settled": ", ".join(sorted(settled)),
            "updates": "; ".join(updates) if updates else "-",
            **{f"d({n})": dist[n] for n in G.nodes()},
        })
    return dist, parent, pd.DataFrame(steps)

dist, parent, steps_df = dijkstra_with_steps(G_city, "Dorm")
steps_df


In [ ]:
def reconstruct_path(parent, target):
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = parent[node]
    return list(reversed(path))

target = "Clinic"  # TODO: change to "Gate"
path_nodes = reconstruct_path(parent, target)
path_edges = list(zip(path_nodes[:-1], path_nodes[1:]))
print("Shortest path:", " -> ".join(path_nodes), "cost =", dist[target])
draw_city_graph(path_edges, title=f"Shortest path from Dorm to {target}")


### Mini exercise

Change the Dorm-Cafe edge from 4 to 12 and rerun Dijkstra. Did the algorithm change, or did your definition of cost change?


In [ ]:
G_city["Dorm"]["Cafe"]["minutes"] = 12
dist2, parent2, steps_df2 = dijkstra_with_steps(G_city, "Dorm")
path_nodes2 = reconstruct_path(parent2, "Clinic")
print("New shortest path:", " -> ".join(path_nodes2), "cost =", dist2["Clinic"])
draw_city_graph(list(zip(path_nodes2[:-1], path_nodes2[1:])), title="After changing one edge weight")


## Part 3: Social network analysis

The Zachary Karate Club network is a classic small social network. Nodes are members. Edges are interactions. The observed label is the group each member joined after the club split.


In [ ]:
K = nx.karate_club_graph()
pos_k = nx.spring_layout(K, seed=4)
clubs = nx.get_node_attributes(K, "club")
colors = ["#2D5BFF" if clubs[n] == "Mr. Hi" else "#E4574E" for n in K.nodes()]

nx.draw_networkx(K, pos_k, node_color=colors, node_size=380, with_labels=True, font_color="white", font_size=8)
plt.title("Zachary Karate Club: two observed groups")
plt.axis("off")
plt.show()


In [ ]:
centrality = pd.DataFrame({
    "degree": nx.degree_centrality(K),
    "betweenness": nx.betweenness_centrality(K),
    "closeness": nx.closeness_centrality(K),
    "eigenvector": nx.eigenvector_centrality(K, max_iter=1000),
})
centrality["club"] = pd.Series(clubs)
centrality.sort_values("betweenness", ascending=False).head(10)


In [ ]:
metric = "betweenness"  # TODO: try degree / closeness / eigenvector
sizes = 300 + 3500 * centrality[metric]
nx.draw_networkx_edges(K, pos_k, alpha=0.25)
nx.draw_networkx_nodes(K, pos_k, node_color=colors, node_size=sizes, edgecolors="#222222")
nx.draw_networkx_labels(K, pos_k, font_color="white", font_size=8)
plt.title(f"Node size = {metric} centrality")
plt.axis("off")
plt.show()


## Part 4: Communities, homophily, and diffusion


In [ ]:
communities = list(nx.community.greedy_modularity_communities(K))
community_id = {}
for i, group in enumerate(communities):
    for node in group:
        community_id[node] = i

same_club_edges = sum(clubs[u] == clubs[v] for u, v in K.edges())
homophily = same_club_edges / K.number_of_edges()

print(f"Detected communities: {len(communities)}")
print(f"Share of edges within the same observed club: {homophily:.2%}")

comm_colors = [community_id[n] for n in K.nodes()]
nx.draw_networkx(K, pos_k, node_color=comm_colors, cmap="Set2", node_size=420, with_labels=True, font_size=8)
plt.title("Greedy modularity communities")
plt.axis("off")
plt.show()


In [ ]:
def threshold_diffusion(G, seeds, threshold=0.30, max_steps=10):
    active = set(seeds)
    history = [set(active)]
    for _ in range(max_steps):
        new_active = set(active)
        for node in G.nodes():
            if node in active:
                continue
            neighbors = list(G.neighbors(node))
            if not neighbors:
                continue
            share_active = sum(n in active for n in neighbors) / len(neighbors)
            if share_active >= threshold:
                new_active.add(node)
        if new_active == active:
            break
        active = new_active
        history.append(set(active))
    return history

seeds = [0]        # TODO: try [0, 33]
threshold = 0.30  # TODO: try 0.2 or 0.5
history = threshold_diffusion(K, seeds, threshold)

plt.plot(range(len(history)), [len(h) for h in history], marker="o")
plt.xlabel("time step")
plt.ylabel("active nodes")
plt.title(f"Threshold diffusion, threshold={threshold}")
plt.show()

print([sorted(list(h)) for h in history])


## Part 5: A tiny two-layer GCN from scratch

We will not use PyTorch Geometric. The goal is to see the message-passing idea directly in matrices.


In [ ]:
A = nx.to_numpy_array(K)
A_tilde = A + np.eye(K.number_of_nodes())
D_inv_sqrt = np.diag(1.0 / np.sqrt(A_tilde.sum(axis=1)))
A_norm = torch.tensor(D_inv_sqrt @ A_tilde @ D_inv_sqrt, dtype=torch.float32)

features_np = np.column_stack([
    [K.degree(n) for n in K.nodes()],
    list(nx.clustering(K).values()),
    list(nx.pagerank(K).values()),
    centrality["betweenness"].values,
]).astype("float32")
features_np = (features_np - features_np.mean(axis=0)) / (features_np.std(axis=0) + 1e-6)
X = torch.tensor(features_np, dtype=torch.float32)

labels_np = np.array([0 if clubs[n] == "Mr. Hi" else 1 for n in K.nodes()])
y = torch.tensor(labels_np, dtype=torch.long)

train_idx = []
for cls in [0, 1]:
    train_idx.extend(np.where(labels_np == cls)[0][:4])
train_idx = torch.tensor(train_idx, dtype=torch.long)
test_idx = torch.tensor([i for i in range(K.number_of_nodes()) if i not in set(train_idx.tolist())], dtype=torch.long)

print("train nodes:", train_idx.tolist())
print("test nodes:", test_idx.tolist()[:10], "...")


In [ ]:
class SimpleGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.lin2 = nn.Linear(hidden_dim, out_dim, bias=False)

    def forward(self, X, A_norm):
        H = torch.relu(A_norm @ self.lin1(X))
        return A_norm @ self.lin2(H)

model = SimpleGCN(in_dim=X.shape[1], hidden_dim=8, out_dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=1e-3)

for epoch in range(201):
    model.train()
    logits = model(X, A_norm)
    loss = F.cross_entropy(logits[train_idx], y[train_idx])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        pred = logits.argmax(dim=1)
        acc = accuracy_score(y[test_idx].numpy(), pred[test_idx].detach().numpy())
        print(f"epoch={epoch:03d} loss={loss.item():.3f} test_acc={acc:.2f}")


In [ ]:
model.eval()
with torch.no_grad():
    pred = model(X, A_norm).argmax(dim=1).numpy()

pred_colors = ["#2D5BFF" if p == 0 else "#E4574E" for p in pred]
nx.draw_networkx(K, pos_k, node_color=pred_colors, node_size=420, with_labels=True, font_color="white", font_size=8)
plt.title("GCN predicted groups")
plt.axis("off")
plt.show()

pd.DataFrame({"node": list(K.nodes()), "true_club": [clubs[n] for n in K.nodes()], "pred": pred}).head(12)


## Part 6: A toy graph agent

This agent does not call an LLM. It demonstrates the core idea: locate concepts in a graph, traverse evidence paths, and return a trace.


In [ ]:
KG = nx.DiGraph()
KG_edges = [
    ("AI and Society", "Networks", "uses relationships to study systems"),
    ("Networks", "Dijkstra", "physical path algorithm"),
    ("Networks", "Social Network Analysis", "position and diffusion"),
    ("Networks", "GNN", "learns on graph structure"),
    ("GNN", "Message Passing", "aggregates neighbors"),
    ("Social Network Analysis", "Centrality", "measures position"),
    ("Social Network Analysis", "Community", "finds group boundaries"),
    ("Dijkstra", "Fairness", "edge weights shape outcomes"),
    ("GNN", "Bias", "relations can propagate bias"),
    ("Graph agent", "Networks", "retrieves evidence along edges"),
    ("Graph agent", "Audit", "records paths and reasons"),
    ("Fairness", "Audit", "checks who is affected"),
    ("Bias", "Audit", "checks error propagation"),
]
KG.add_edges_from((u, v, {"relation": r}) for u, v, r in KG_edges)

def graph_agent(start, goal):
    undirected = KG.to_undirected()
    path = nx.shortest_path(undirected, start, goal)
    steps = []
    for u, v in zip(path[:-1], path[1:]):
        rel = KG.get_edge_data(u, v, default=KG.get_edge_data(v, u, default={})).get("relation", "related")
        steps.append({"from": u, "to": v, "edge_meaning": rel})
    return path, pd.DataFrame(steps), " -> ".join(path)

path, trace, explanation = graph_agent("Dijkstra", "Audit")
print("Agent path:", explanation)
trace


In [ ]:
pos_kg = nx.spring_layout(KG.to_undirected(), seed=2)
path_edges = set(zip(path[:-1], path[1:])) | set(zip(path[1:], path[:-1]))
edge_colors = ["tomato" if (u, v) in path_edges or (v, u) in path_edges else "#BBBBBB" for u, v in KG.edges()]
node_colors = ["#F6D365" if n in path else "#E8EEF7" for n in KG.nodes()]

nx.draw_networkx(KG, pos_kg, node_color=node_colors, edge_color=edge_colors, node_size=1250, arrows=True, font_size=8)
edge_labels = nx.get_edge_attributes(KG, "relation")
nx.draw_networkx_edge_labels(KG, pos_kg, edge_labels=edge_labels, font_size=7)
plt.title("Graph agent trace")
plt.axis("off")
plt.show()


## Reflection

1. If an edge weight represents travel time, whose time is counted?
2. Should central nodes receive more resources, more scrutiny, or more protection?
3. If a GNN learns historical bias, through which edges can that bias spread?
4. A graph-agent path is explainable, but who audits the choice of path?
